# *<center>V06 · Transport and the Einstein relation</center>*

**Purpose.** Validate ion transport in gas: drift velocity linear in
field (mobility K), diffusion from mean-square displacement (D), and the
**Einstein closure D/K = kT/q** — the fluctuation-dissipation statement
that ties V05's thermal statistics to directed transport. HS is tested
against kinetic theory and the closure; **SDS is tested against itself**
(its native validation, deferred here from V05), and its measured
departure from the Einstein ratio is characterized with a declared
operating point rather than absorbed into a loose threshold.

```
PROVENANCE
  origin   : validation series
  template : V01/V05 (assumptions / methods / citations; empirical bands)
```

### Conventions
* **Units are mm, V, µs, K**; K in mm²/(V·µs) (numerically equal to
  m²/(V·s)); D in mm²/µs.
* **CAPITALS are parameters you may change**; lower-case is computed.
* Thresholds are declared before each measurement and asserted; bands
  are calibrated from **measured seed-to-seed scatter** (the V05
  lesson), never from sqrt(2/N) alone.

---

### Assumptions (explicit)
1. **Uniform field from the V01-certified geometry.** The drift cell is
   the parallel-plate box V01 proved uniform to machine precision; the
   field magnitude is V/gap, asserted against the spec before flying
   (a silent voltage-wiring failure must fail loudly, not fly at E=0).
2. **Low-field regime.** At the operating point (N2, 2 Torr, 300 K,
   m/z 100), E/N runs ~8-31 Td across the drift ladder; drift stays
   well below thermal speed, and measured K is field-flat to a few
   percent — the linear-transport regime the closure assumes [1].
3. **Lag-averaged MSD.** D comes from the time-origin-averaged MSD
   slope over lags 1-8 µs (every trajectory contributes ~all its time
   origins). Single-origin MSD at these ensemble sizes scatters ~35%
   seed-to-seed; lag averaging brings it to ~8-16% (measured), and the
   bands below carry that number.
4. **Sampled, not mean** *(amended)*:
   HS draws every stochastic element from its distribution — free
   paths exponential via the per-step Poisson test with the
   speed-dependent λ(v) (Maxwell relative-speed correction) as rate
   parameter; collision partners from the **relative-speed-weighted**
   Maxwellian (rejection sampling — collision-flux weighting, not the
   mean gas velocity and not plain MB); impact geometry uniform over
   the impact-parameter disc. **SDS is deliberately different**: a bulk
   model with deterministic mobility damping plus diffusion jumps
   **sampled from the Appelhans–Dahl ICDF tables** (`sds_jump_icdf.dat`
   — ion_gym's own regeneration of the published procedure, adopted
   This notebook's 4/4 re-execution on the regenerated
   tables is the functional proof;
   inverse-transform draw, log-interpolated between the bracketing
   mass-ratio decade curves — continuous in mass ratio) with the jump
   direction sampled uniformly; but the **collision count per step is
   the mean** (√(V̄Δt/λ̄·10⁻⁵) scaling, no Poisson draw — negligible in
   the many-collisions-per-step design regime) and the ion carries **no
   thermal velocity** (fluctuation enters as displacement only) — no
   discrete partners, no individual free paths. That structural difference is why SDS's
   Einstein-ratio departure (assumption 6 below) is a model-level
   property, not a sampling artifact.
5. **The planar cell is the 2-D slice** (V05): its Einstein closure is
   evaluated at the slice's OWN measured temperature (m·Var(v)/kB on
   the equilibrated tail), which V05 showed sits ~4% below the bath.
6. **SDS operating point.** SDS results are quoted at dt = 5 ns AND
   cross-checked in the model's design regime (dt = 100 ns, ~8.7
   collisions/step, per its atmospheric-pressure lineage [5]): the
   Einstein-ratio departure is dt-ROBUST (2.18 at 5 ns, 2.25 at
   100 ns) and mass-flat, and the departure is located in the MODEL
   itself rather than in this implementation (see the SDS section).
   D_sds varies ~10% across dt, within the lag-MSD scatter.
7. **Survivor statistics.** Fits use ions that survive to timeout;
   survival is printed with every measurement.

### Numerical methods (explicit)
* **Drift**: per-ion linear fit of y(t) over the post-transient window;
  K = mean slope / E; linearity checked across a 4x field ladder.
* **Diffusion**: lag-averaged MSD (assumption 3), slope/2 per axis.
* **Kinetic-theory reference**: first Chapman-Enskog approximation for
  hard spheres, K_CE = (3q/16N)·sqrt(2*pi/(mu*k*T))/sigma [1, 2]. With
  dt resolving the collision rate (nu*dt = 0.09 at dt = 1 ns), measured
  K agrees with K_CE to ~1.4%.
* **The nu*dt criterion (a quantified artifact).** The HS kernel
  applies AT MOST ONE collision per step (probability 1-exp(-nu*dt)),
  which under-counts multi-collision steps and inflates K when nu*dt is
  not small: at 2 Torr, K runs +16% high at dt = 5 ns (nu*dt = 0.43),
  +7% at 2 ns, +1.4% at 1 ns — the convergence is shown and asserted
  below. The Einstein RATIO is blind to this (D and K carry the same
  rate deficit, which cancels) — a strength of the closure test and,
  equally, its blind spot for absolute coefficients.
* **Einstein closure**: (D/K)·(q/kT) -> 1 [4]; for the planar slice, T
  is the slice's measured temperature (assumption 4).

### Citations
1. E. A. Mason, E. W. McDaniel, *Transport Properties of Ions in
   Gases*, Wiley (1988) — mobility, diffusion, low-field criteria, Td.
2. S. Chapman, T. G. Cowling, *The Mathematical Theory of Non-Uniform
   Gases*, 3rd ed., CUP (1970) — the CE hard-sphere first approximation.
3. A. Einstein, *Ann. Phys.* 17, 549 (1905) — the D/K = kT/q relation.
5. A. D. Appelhans, D. A. Dahl, *Int. J. Mass Spectrom.* 244, 1-14
   (2005) — the SDS model (Stokes-damped drift + statistical diffusion).
6. F. Reif, *Fundamentals of Statistical and Thermal Physics* (1965) —
   random walks, fluctuation-dissipation.


## The instrument, before any statistics

The device this notebook flies, drawn from the **solver's own electrode mask** (not a redrawing) with example ion paths exactly as flown. You are looking at the uniform-field tube used as the mobility referee, with example ions drifting at constant average velocity through the gas — the spreading you see IS the diffusion the Einstein relation predicts.

Deck: `examples/drift_tube_mason_schamp.json`. A geometry figure is not decoration — if the picture and the solved model can disagree, every number below is unverifiable.

In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/drift_tube_mason_schamp.json', banked='panel_drift_tube.png', height=520)


In [ ]:
import math
import os
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import sys
from IPython.display import display
sys.path.insert(0, '..')

from ion_gym.io.sim_spec import (SimSpec, GeometrySpec, ElectrodeSpec,
                                 ShapeSpec, SourceSpec, IntegrationSpec,
                                 BoundsSpec, CollisionSpec)
from ion_gym.physics.symmetry import SymmetrySpec
from ion_gym.physics.sim_build import build_run
from ion_gym.physics.traj_stats import var_temperature

KB = 1.380649e-23
AMU = 1.66053906660e-27
E_CHG = 1.602176634e-19
MMUS = 1.0e3

# ---- parameters (CAPITALS are yours to change) ----
T_GAS_K = 300.0
P_TORR = 2.0
MZ = 100.0
SEED = 11
GAP_MM = 9.0                    # plate gap of the V01-certified cell
E_LADDER_VMM = [0.5, 1.0, 2.0]  # drift fields
N_DRIFT = 32
N_DIFF = 96
T_DRIFT_US = 40.0
T_DIFF_US = 30.0
m_kg = MZ * AMU
m_gas_amu = 28.0134
SIGMA_HS_M2 = 2.27e-18          # the HS kernel's hard-sphere sigma

DT_HS_NS = 1.0                 # nu*dt = 0.09: resolves the collision rate

def gas_box(v_top, n_ions, t_max, y0, t_start_k=T_GAS_K, seed=SEED,
            model="hs", dt_ns=None):
    W = 10.0
    H = 10.0
    t = 0.5
    geom = GeometrySpec(
        width_mm=W, height_mm=H, mm_per_gu=0.1,
        symmetry=SymmetrySpec(coords="xyz"),
        electrodes=[
            ElectrodeSpec(name="bot", dc=0.0, shapes=[ShapeSpec(
                "rect", {"x_mm": 0.0, "y_mm": 0.0,
                         "width_mm": W, "height_mm": t})]),
            ElectrodeSpec(name="top", dc=v_top, shapes=[ShapeSpec(
                "rect", {"x_mm": 0.0, "y_mm": H - t,
                         "width_mm": W, "height_mm": t})])])
    sp = SimSpec(name="V06 cell", geometry=geom,
                 source=SourceSpec(n_ions=n_ions, distribution="point",
                                   x0_mm=W / 2, y0_mm=y0, mz_list=[MZ],
                                   seed=seed, temperature_k=t_start_k),
                 integration=IntegrationSpec(
                     t_max_us=t_max,
                     dt_ns=(DT_HS_NS if dt_ns is None else dt_ns),
                     rec_every=int(100 / (DT_HS_NS if dt_ns is None
                                          else dt_ns))),
                 bounds=BoundsSpec(),
                 collisions=CollisionSpec(enabled=True, gas="N2",
                                          T_k=T_GAS_K, model=model))
    sp.collisions.set_pressure_torr(P_TORR)
    return sp

def fly_all(sp):
    errs = sp.validate()
    assert not errs, errs
    model, f, cols, births = build_run(sp)
    res = [f(i) for i in range(len(births))]
    return res, {cc: j for j, cc in enumerate(cols)}

def drift_K(res, ci, E_vmm, t_min_us=5.0):
    slopes = []
    for tr, s in res:
        if tr is None or len(tr) < 20 or s["kind"] != 2:
            continue
        m = tr[:, ci["t"]] > t_min_us
        slopes.append(np.polyfit(tr[m, ci["t"]], tr[m, ci["y"]], 1)[0])
    slopes = np.array(slopes)
    sem = slopes.std(ddof=1) / math.sqrt(len(slopes))
    return slopes.mean() / E_vmm, sem / E_vmm, len(slopes)

def lag_msd_D(res, ci, ch, lag_lo_us=1.0, lag_hi_us=8.0, rec_dt_us=0.1):
    lags = np.arange(int(lag_lo_us / rec_dt_us),
                     int(lag_hi_us / rec_dt_us) + 1, 4)
    taus = lags * rec_dt_us
    msd = np.zeros(len(lags))
    wts = np.zeros(len(lags))
    n_used = 0
    for tr, s in res:
        if tr is None or len(tr) < lags[-1] + 8:
            continue
        n_used += 1
        x = tr[:, ci[ch]]
        for k, L in enumerate(lags):
            d = x[L:] - x[:-L]
            msd[k] += np.sum(d * d)
            wts[k] += len(d)
    msd = msd / wts
    D = np.polyfit(taus, msd, 1)[0] / 2.0
    return D, taus, msd, n_used

N_dens = P_TORR * 133.322 / (KB * T_GAS_K)
mu = (MZ * m_gas_amu) / (MZ + m_gas_amu) * AMU
K_CE = (3 * E_CHG / (16 * N_dens)
        * math.sqrt(2 * math.pi / (mu * KB * T_GAS_K)) / SIGMA_HS_M2)
K_CE = K_CE * 1.0               # m^2/(V s) == mm^2/(V us) numerically
print(f"operating point: N2 {P_TORR:g} Torr {T_GAS_K:g} K, m/z {MZ:g}; "
      f"E/N ladder = "
      f"{[round(E * 1e3 / N_dens / 1e-21, 1) for E in E_LADDER_VMM]} Td")
print(f"Chapman-Enskog hard-sphere reference K_CE = {K_CE:.4f} mm^2/(V us)")

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


## HS — drift: linearity, mobility, and the nu*dt convergence

Three fields spanning 4x at dt = 1 ns (nu*dt = 0.09), plus a
dt-convergence ladder at E = 1 V/mm showing the one-collision-per-step
artifact decaying (assumption above).

**Pass thresholds:** each field's K within **max(5%, 2·sem)** of the
pooled K — the lowest rung's own statistical error (~9% at 32 ions:
slow drift against diffusion noise) exceeds a flat 5%, so the band is
per-rung, calibrated to the measurement (the V05 doctrine); pooled K
inside **[0.95, 1.10]·K_CE**; the dt ladder monotone:
K(5 ns) > K(2 ns) > K(1 ns).

In [ ]:
K_by_E = {}
for E in E_LADDER_VMM:
    v_top = -E * GAP_MM
    sp = gas_box(v_top, N_DRIFT, T_DRIFT_US, y0=2.0)
    # FIELD GUARD (assumption 1): the spec must actually carry the drive
    els = {e.name: e for e in sp.geometry.electrodes}
    assert abs(els["top"].dc - v_top) < 1e-12, "drive not on the spec!"
    res, ci = fly_all(sp)
    K, K_sem, n = drift_K(res, ci, E)
    K_by_E[E] = (K, K_sem, n)
    print(f"  E = {E:4.1f} V/mm: K = {K:.4f} ± {K_sem:.4f} "
          f"mm^2/(V us)  [{n}/{N_DRIFT} ions]")
K_pool = float(np.mean([K for K, _, _ in K_by_E.values()]))
lin_ok = all(abs(K - K_pool) < max(0.05 * K_pool, 2.0 * K_sem)
             for K, K_sem, _ in K_by_E.values())
ratio_ce = K_pool / K_CE
print(f"  pooled K = {K_pool:.4f} mm^2/(V us); K/K_CE = {ratio_ce:.3f}")

# the dt-convergence ladder: the artifact, shown decaying
K_dt = {}
for dt in (5.0, 2.0, 1.0):
    sp = gas_box(-1.0 * GAP_MM, N_DRIFT, T_DRIFT_US, y0=2.0, dt_ns=dt)
    res, ci = fly_all(sp)
    K_dt[dt], _, _ = drift_K(res, ci, 1.0)
    nudt = 0.087 * dt
    print(f"  dt = {dt:3.0f} ns (nu*dt = {nudt:.2f}): "
          f"K/K_CE = {K_dt[dt] / K_CE:.3f}")
PASS_DRIFT = (lin_ok and (0.95 < ratio_ce < 1.10)
              and K_dt[5.0] > K_dt[2.0] > K_dt[1.0])
print("PASS" if PASS_DRIFT else "FAIL",
      "— drift linear; K agrees with Chapman-Enskog at resolved dt; "
      "the one-collision-per-step artifact decays monotonically")
assert PASS_DRIFT

### The two terms this notebook leans on: MSD and τ

**MSD — mean-squared displacement.** Take an ion, note where it is, wait a while, note where it is again, and square the distance it moved along one axis. Average that over many ions (and, below, over many starting instants). That average is the MSD. It is the natural measure of *spreading*, because a diffusing packet has zero mean displacement — as many ions go left as right — while the squared displacement grows steadily. For ordinary diffusion the growth is linear:

  ⟨Δx²(τ)⟩ = 2·D·τ

so **D is half the slope of MSD against τ**, and that slope is exactly what the cells below fit.

**τ — the lag time.** τ is *not* a fixed property of the ion or a fitted time constant. It is the **elapsed time between the two positions you compare** — the separation between samples, also called the lag. Plotting MSD against τ means: "how far has an ion typically drifted apart from where it was, τ microseconds earlier?" The x-axis of the MSD plot is that τ.

**Why "lag-averaged".** Rather than measuring every displacement from the moment of birth, the code slides a window along each trajectory and uses *every* pair of samples separated by τ, then averages. This uses far more of each trajectory (better statistics from the same flight) and removes any dependence on the birth instant, which may sit in a transient before the ions have thermalized.

**How to read the plot.** A straight line through the origin means normal diffusion, and its slope gives D. Curvature at **small τ** means the ion has not yet forgotten its initial velocity — below roughly one collision interval, motion is still ballistic (MSD ∝ τ², steeper). Flattening at **large τ** means too few independent sample pairs are left, or ions are reaching a wall. The fit window (`lag_lo_us`, `lag_hi_us`) is chosen to sit between those two failure modes, and the point counts printed alongside are how you check that it does.

## HS — diffusion and the Einstein closure (at the slice's own T)

Field-free cell, lag-averaged MSD per in-plane axis. The closure uses
the slice's MEASURED temperature (assumption 4, V05).

**Pass thresholds:** D_x and D_y within **20%** of each other
(isotropy at the measured scatter); Einstein ratio
(D_mean/K_pool)·(q/(k·T_slice)) inside **[0.80, 1.25]** — the band
carries the ~8-16% lag-MSD seed scatter plus the K and T errors.

In [ ]:
sp0 = gas_box(0.0, N_DIFF, T_DIFF_US, y0=5.0)
res0, ci0 = fly_all(sp0)
# the slice's own temperature from the equilibrated tail (V05 machinery)
pool = []
for tr, s in res0:
    if tr is None or len(tr) < 20:
        continue
    m = tr[:, ci0["t"]] > 0.5 * T_DIFF_US
    pool.append(tr[m, ci0["vx"]])
    pool.append(tr[m, ci0["vy"]])
T_slice = var_temperature(np.concatenate(pool), m_kg)
D_x, taus, msd_x, n_x = lag_msd_D(res0, ci0, "x")
D_y, _, msd_y, _ = lag_msd_D(res0, ci0, "y")
D_mean = 0.5 * (D_x + D_y)
einstein = (D_mean / K_pool) * (E_CHG / (KB * T_slice))
print(f"  T_slice = {T_slice:.1f} K (V05: the slice runs ~4% cold)")
print(f"  D_x = {D_x:.5f}, D_y = {D_y:.5f} mm^2/us  [{n_x} ions]")
print(f"  Einstein ratio (D/K)/(kT_slice/q) = {einstein:.3f}")
PASS_EIN = (abs(D_x - D_y) < 0.20 * D_mean
            and 0.80 < einstein < 1.25)
print("PASS" if PASS_EIN else "FAIL",
      "— in-plane D isotropic; Einstein closure holds at the slice's "
      "own temperature")
assert PASS_EIN

fig, axes = plt.subplots(1, 2, figsize=(8, 6))
ts_d = np.linspace(0, T_DRIFT_US, 50)
for E in E_LADDER_VMM:
    K, _, _ = K_by_E[E]
    axes[0].plot(ts_d, K * E * ts_d, "--", lw=1,
                 label=f"E={E:g}: K·E·t")
axes[0].set_xlabel("t (us)")
axes[0].set_ylabel("mean drift displacement (mm)")
axes[0].set_title(f"drift ladder (pooled K = {K_pool:.4f})", fontsize=9)
axes[0].legend(fontsize=7)
axes[1].plot(taus, msd_x, "o", ms=3, label="MSD_x (lag-averaged)")
axes[1].plot(taus, msd_y, "s", ms=3, label="MSD_y")
axes[1].plot(taus, 2 * D_mean * taus, "k-", lw=1.2,
             label=f"2·D·tau, D = {D_mean:.5f}")
axes[1].set_xlabel("lag tau (us)")
axes[1].set_ylabel("MSD (mm^2)")
axes[1].set_title(f"diffusion; Einstein ratio {einstein:.2f}", fontsize=9)
axes[1].legend(fontsize=7)
fig.suptitle(f"V06 HS transport at {P_TORR:g} Torr N2, m/z {MZ:g}, "
             f"T_slice {T_slice:.0f} K", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.94))
display(fig)
plt.close(fig)

## SDS — self-consistency at a declared operating point

The SDS model (Stokes-damped drift + table-based positional diffusion,
no thermal velocity content) is flown through the 3-D import path on the
tiny grounded box, one plate driven. Two findings are characterized
here rather than hidden:

* its **mobility is realistic** — K_sds ≈ 0.073-0.074 mm²/(V·µs) at
  this point, within ~7% of the empirical reduced-mobility expectation
  for an m/z-100 ion in N2 — and seed-stable to ~3%;
* its **diffusion does NOT close the Einstein relation**: measured
  D/K ≈ **2.1-2.3 × kT/q**, dt-robust (5-100 ns) and mass-flat
  (m/z 28-400). The implementation is internally consistent
  (sphere_rand, Stokes damping, ICDF interpolation, the diffusion
  chain, all constants), and the departure reproduces ANALYTICALLY
  from the model's own pieces — D implied by the displacement
  tables + parameters is 0.00412 vs
  0.00419 measured (kernel faithful to ~2%) vs 0.00197
  Einstein-implied: **table/Einstein = 2.09 with no flight involved**.
  The cause is the MODEL: empirical-Ko drift (real, polarization-
  enhanced collisions) paired with geometric hard-sphere displacement
  tables (mass-fit diameters) — fluctuation-dissipation closure
  between them was never enforced [5]. SDS spreading estimates carry
  this factor by construction.

**Pass thresholds (characterization bands):** K_sds in
**[0.060, 0.090]**; Einstein ratio in **[1.8, 2.8]** at dt = 5 ns —
these assert the DOCUMENTED behavior so any silent change to the SDS
implementation fails this notebook.

In [ ]:
# The 3-D case: a tiny grounded-plate box authored NATIVELY as two
# extruded rects and flown through the ordinary build_run dispatch.
# (Re-rooted off the retired import path; equivalence to
# the geometry it replaces was proven first on the smallest falsifying
# system -- identical 21x17x13 grid at 0.5 mm, identical plate extents,
# identical fill 11.76% per plate / 23.53% total.)
BOX3D_W_MM, BOX3D_H_MM, BOX3D_D_MM = 10.0, 8.0, 6.0   # outer box [mm]
BOX3D_PLATE_T_MM = 0.5                                 # plate thickness [mm]
BOX3D_PITCH_MM = 0.5                                   # raster pitch [mm]

def box3d_geometry(pitch_mm=BOX3D_PITCH_MM):
    """Two grounded plates facing across a 7 mm gap; open in x and z."""
    ex = {"axis": "z", "lo_mm": 0.0, "hi_mm": BOX3D_D_MM}
    plate = lambda nm, y0: ElectrodeSpec(name=nm, dc=0.0, shapes=[ShapeSpec(
        "rect", {"x_mm": 0.0, "y_mm": y0, "width_mm": BOX3D_W_MM,
                 "height_mm": BOX3D_PLATE_T_MM, "extrude": dict(ex)})])
    return GeometrySpec(
        width_mm=BOX3D_W_MM, height_mm=BOX3D_H_MM, depth_mm=BOX3D_D_MM,
        mm_per_gu=pitch_mm, symmetry=SymmetrySpec(coords="xyz"),
        electrodes=[plate("bottom", 0.0),
                    plate("top", BOX3D_H_MM - BOX3D_PLATE_T_MM)])
V_PLATE = -8.0
E_SDS = -V_PLATE / 7.0            # 7 mm inner gap of the box

def sds_case(v2, n_ions, y0, seed):
    """Drive the TOP plate to v2 and fly SDS across the 7 mm gap."""
    geom = box3d_geometry()
    for e in geom.electrodes:
        if e.name == "top":
            e.dc = v2
    sp = SimSpec(name="V06 3-D box (native)", geometry=geom,
                 source=SourceSpec(n_ions=n_ions, distribution="point",
                                   x0_mm=5.0, y0_mm=y0, z0_mm=3.0,
                                   mz_list=[MZ], seed=seed,
                                   temperature_k=T_GAS_K),
                 integration=IntegrationSpec(t_max_us=25.0, dt_ns=5.0,
                                             rec_every=20,
                                             record_channels=["speed",
                                                              "ke_ev"]),
                 bounds=BoundsSpec(),
                 collisions=CollisionSpec(enabled=True, gas="N2",
                                          T_k=T_GAS_K, model="sds"))
    sp.collisions.set_pressure_torr(P_TORR)
    # FIELD GUARD: the driven plate must actually carry its volts
    drv = [e for e in sp.geometry.electrodes if abs(e.dc - v2) < 1e-12]
    assert v2 == 0.0 or drv, "SDS drive not on the spec!"
    m3, f3, cols3, b3 = build_run(sp)
    ci = {cc: j for j, cc in enumerate(cols3)}
    return [f3(i) for i in range(len(b3))], ci

res_d, ci_d = sds_case(V_PLATE, 24, 2.0, SEED)
K_sds, K_sem, n_k = drift_K(res_d, ci_d, E_SDS, t_min_us=3.0)
res_0, ci_0 = sds_case(0.0, 48, 4.0, SEED)
D_sds, taus_s, msd_s, n_s = lag_msd_D(res_0, ci_0, "x")
einstein_sds = (D_sds / K_sds) * (E_CHG / (KB * T_GAS_K))
print(f"  K_sds = {K_sds:.4f} ± {K_sem:.4f} mm^2/(V us)  "
      f"[{n_k} ions, E = {E_SDS:.3f} V/mm]")
print(f"  D_sds = {D_sds:.5f} mm^2/us at dt = 5 ns  [{n_s} ions]")
print(f"  Einstein ratio (D/K)/(kT/q) = {einstein_sds:.2f}  "
      f"(HS closes at ~1; SDS characterized at ~2.2-2.4 here)")
PASS_SDS = (0.060 < K_sds < 0.090) and (1.8 < einstein_sds < 2.8)
print("PASS" if PASS_SDS else "FAIL",
      "— SDS behaves as characterized at the declared operating point "
      "(mobility realistic; Einstein departure documented + logged)")
assert PASS_SDS

## r-z cross-check — diffusion isotropy on the full-3-D kernel

Field-free tube (V05 geometry), lag-MSD per axis. **Pass threshold:**
each axis within **25%** of the three-axis mean (isotropy at these
sample sizes; the empirical lag-MSD scatter is ~8-16% per axis).

In [ ]:
def rz_tube(n_ions, t_max, seed):
    geom = GeometrySpec(
        width_mm=10.0, height_mm=5.0, mm_per_gu=0.1,
        symmetry=SymmetrySpec(coords="rz"),
        electrodes=[ElectrodeSpec(name="tube", dc=0.0, shapes=[ShapeSpec(
            "rect", {"x_mm": 0.0, "y_mm": 4.5, "width_mm": 10.0,
                     "height_mm": 0.5})])])
    sp = SimSpec(name="V06 rz", geometry=geom,
                 source=SourceSpec(n_ions=n_ions, distribution="point",
                                   x0_mm=5.0, y0_mm=1.0, mz_list=[MZ],
                                   seed=seed, temperature_k=T_GAS_K),
                 integration=IntegrationSpec(t_max_us=t_max, dt_ns=5.0,
                                             rec_every=20,
                                             record_channels=["speed",
                                                              "ke_ev"]),
                 bounds=BoundsSpec(),
                 collisions=CollisionSpec(enabled=True, gas="N2",
                                          T_k=T_GAS_K, model="hs"))
    sp.collisions.set_pressure_torr(P_TORR)
    return sp
res_rz, ci_rz = fly_all(rz_tube(48, 30.0, SEED))
D_ax = {}
for ch in ("x", "y", "z"):
    D_ax[ch], _, _, n_rz = lag_msd_D(res_rz, ci_rz, ch)
D_bar = float(np.mean(list(D_ax.values())))
for ch, D in D_ax.items():
    print(f"  D_{ch} = {D:.5f} mm^2/us ({(D / D_bar - 1) * 100:+.0f}% "
          f"of mean)  [{n_rz} ions]")
PASS_RZ = all(abs(D - D_bar) < 0.25 * D_bar for D in D_ax.values())
print("PASS" if PASS_RZ else "FAIL",
      "— diffusion isotropic on the full-3-D kernel")
assert PASS_RZ

---
### What this notebook established
* **HS drift is linear** across a 4x field ladder and its mobility
  agrees with the first Chapman-Enskog approximation to **~1.4%** once
  dt resolves the collision rate; the one-collision-per-step artifact
  (+16% at nu*dt = 0.43) is quantified and shown converging away — a
  dt criterion now stated for all HS transport work.
* **The Einstein relation closes** on HS at the slice's own measured
  temperature — the fluctuation (V05) and dissipation (this notebook)
  sides of the same kernel agree.
* **SDS is characterized, not assumed**: realistic mobility (~7% from
  the empirical expectation), but diffusion ~2.1x the
  Einstein-consistent value — the departure is the MODEL's
  (empirical-Ko drift vs hard-sphere displacement tables, Einstein
  closure never enforced between them). The assertion bands pin this
  documented behavior so silent changes to the implementation fail
  here.
* **Diffusion is isotropic** on the full-3-D kernel.

**Next in the series:** V03 — pseudopotential vs direct RF integration
(where the adiabatic approximation holds and where it degrades).

## Read-out

The Einstein relation ties mobility to diffusion through the temperature: a transport model that gets drift velocity right but diffusion wrong will produce plausible arrival times with the wrong peak widths. Closing this relation is what licenses using the model for resolution work at all.